# 05b - (Tahap 1) Cleaning dataset with Rule-based

In [1]:
!pip install -q sentence-transformers faiss-cpu huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.9 MB/s eta 0:00:00


In [2]:
import json, os, re, hashlib, difflib, datetime
import numpy as np
import faiss
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient
import torch
import random

## 1. Konfigurasi & Load

In [3]:
CONFIG = {
    "input_dir":               "/kaggle/input/notebooks/muhammadnabilul/05-generating-qa/output_05a",
    "output_dir":              "output_05b",
    "hf_repo_id":              "Makaareeem/publikasi-rag-knowledge-base",
    "embedding_model":         "nomic-ai/nomic-embed-text-v1.5",
    "dedup_threshold":         0.97,
    "dedup_max_per_cluster":   2,
    "degenerate_min_words":    6,
    "chunk_id_hard_fail_ratio": 0.5,
    "embed_batch_size":        128,
    "preview_n":               4,
    "preview_seed":            42,
    "dedup_seed":              42,
}
 
ALLOWED_TASKS = {"qa", "summary"}
ALLOWED_SUBTASKS = {
    "factual_retrieval", "source_navigation",
    "definition_clarification", "methodology_explanation",
    "trend_interpretation", "causal_reasoning",
    "comparative_analysis", "multi_source_synthesis",
    "document_overview",
}
 
# Hard gate: instruction merujuk langsung ke tabel/konteks
TABLE_REF_PATTERNS = [
    r"dalam tabel ini",
    r"dalam konteks tabel ini",
    r"dalam tabel berikut",
    r"tabel di atas",
    r"dalam tabel tersebut",
]
 
# Awalan meta-reference yang perlu di-STRIP (bukan dibuang)
META_REF_AWALAN = [
    r"^(berdasarkan|menurut|sesuai|dalam)\s+(teks sumber|passages|konteks|dokumen ini)[,.]?\s*",
    r"^(teks sumber|passages|konteks|dokumen ini)\s+(ini\s+)?menunjukkan[,.]?\s*",
    r"^(teks sumber|passages|konteks|dokumen ini)\s+(ini\s+)?menyatakan[,.]?\s*",
    r"^(teks sumber|passages|konteks|dokumen ini)\s+(ini\s+)?menjelaskan[,.]?\s*",
]
 
# Soft score: frasa yang perlu dilacak tapi tidak dibuang
LEAK_HIGH_CONFIDENCE = [
    r"teks sumber ini",
    r"\bpassages\b",
    r"konteks ini",
    r"bagian ini",
    r"dokumen ini",
    r"di atas",
    r"\(tidak disebut eksplisit",
    r"\(dihitung dari",
]
 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
def load_raw_records(input_dir):
    path = os.path.join(input_dir, "raw_records.jsonl")
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records
 
raw_records = load_raw_records(CONFIG["input_dir"])
print(f"Loaded {len(raw_records)} raw records")

Loaded 24426 raw records


## 2. Fungsi-Fungsi

### a. Bersihkan teks

In [5]:
ARTIFACT_PATTERN = re.compile(r"[\xa0\u200b\u200c\u200d\ufeff]|<[^>]+>")
 
def clean_whitespace(text):
    if not text:
        return text
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()
 
n_artifact = 0
for r in raw_records:
    had = ARTIFACT_PATTERN.search(r.get("instruction","") + r.get("response",""))
    r["instruction"] = clean_whitespace(r.get("instruction",""))
    r["response"]    = clean_whitespace(r.get("response",""))
    if had:
        n_artifact += 1
 
print(f"Artefak unicode/HTML dibersihkan: {n_artifact}")

Artefak unicode/HTML dibersihkan: 35


### b. Cleaning

In [6]:
def strip_meta_awalan(text):
    original = text
    for pat in META_REF_AWALAN:
        text = re.sub(pat, "", text, flags=re.IGNORECASE).strip()
    if text and text != original:
        text = text[0].upper() + text[1:]
    return text, text != original
 
n_cleaned = 0
for r in raw_records:
    r["flags"] = {}
    new_resp, changed = strip_meta_awalan(r.get("response",""))
    if changed:
        r["response"] = new_resp
        r["flags"]["meta_ref_cleaned"] = True
        n_cleaned += 1
 
print(f"Meta-reference awalan di-strip: {n_cleaned} records")

Meta-reference awalan di-strip: 24 records


### c.Valid chunks dari HF

In [7]:
secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_Faiss"))
meta_path = hf_hub_download(
    repo_id=CONFIG["hf_repo_id"],
    filename="chunk_metadata.jsonl",
    repo_type="dataset"
)
with open(meta_path, encoding="utf-8") as f:
    hf_meta = [json.loads(line) for line in f]
VALID_CHUNK_IDS = set(c["chunk_id"] for c in hf_meta)
print(f"VALID_CHUNK_IDS: {len(VALID_CHUNK_IDS)}")

chunk_metadata.jsonl:   0%|          | 0.00/43.9M [00:00<?, ?B/s]

VALID_CHUNK_IDS: 23158


## d. Batas Utama

In [8]:
def check_schema(r):
    if r.get("task_type") not in ALLOWED_TASKS:
        return False, f"task_type invalid: {r.get('task_type')}"
    if r.get("subtask_type") not in ALLOWED_SUBTASKS:
        return False, f"subtask_type invalid: {r.get('subtask_type')}"
    return True, None
 
def check_chunk_ids(r):
    ids = r.get("context_chunk_ids", [])
    if not ids:
        return False, "context_chunk_ids kosong", []
    missing = [cid for cid in ids if cid not in VALID_CHUNK_IDS]
    if not missing:
        return True, None, []
    ratio = len(missing) / len(ids)
    if ratio >= CONFIG["chunk_id_hard_fail_ratio"]:
        return False, missing, missing
    return True, None, missing
 
def check_degenerate(r):
    resp = (r.get("response") or "").strip()
    if not resp or len(resp.split()) < CONFIG["degenerate_min_words"]:
        return False, "response kosong/terlalu pendek"
    return True, None
 
def check_table_ref(r):
    instr = (r.get("instruction") or "").lower()
    for pat in TABLE_REF_PATTERNS:
        if re.search(pat, instr, re.IGNORECASE):
            return False, f"instruction merujuk tabel langsung: '{pat}'"
    return True, None
 
def apply_hard_gates(records):
    for r in records:
        ok, reason = check_schema(r)
        if not ok:
            r["flags"]["schema_invalid"] = reason
 
        ok, reason, missing = check_chunk_ids(r)
        if missing:
            r["flags"]["chunk_id_partial_invalid"] = True
            r["flags"]["missing_chunk_ids"] = missing
        if not ok:
            r["flags"]["chunk_id_invalid"] = missing
 
        ok, reason = check_degenerate(r)
        if not ok:
            r["flags"]["degenerate"] = reason
 
        ok, reason = check_table_ref(r)
        if not ok:
            r["flags"]["table_ref_instruction"] = reason
 
    return records
 
raw_records = apply_hard_gates(raw_records)
print(f"schema_invalid          : {sum(1 for r in raw_records if 'schema_invalid' in r['flags'])}")
print(f"chunk_id_invalid        : {sum(1 for r in raw_records if 'chunk_id_invalid' in r['flags'])}")
print(f"degenerate              : {sum(1 for r in raw_records if 'degenerate' in r['flags'])}")
print(f"table_ref_instruction   : {sum(1 for r in raw_records if 'table_ref_instruction' in r['flags'])}")
print(f"chunk_id_partial_invalid: {sum(1 for r in raw_records if 'chunk_id_partial_invalid' in r['flags'])}")

schema_invalid          : 0
chunk_id_invalid        : 0
degenerate              : 20
table_ref_instruction   : 501
chunk_id_partial_invalid: 0


### e. Near duplicate

In [9]:
embed_model = SentenceTransformer(
    CONFIG["embedding_model"],
    trust_remote_code=True,
    device=DEVICE,
)
 
texts = [f"search_document: {r['instruction']} {r['response']}" for r in raw_records]
embeddings = embed_model.encode(
    texts,
    batch_size=CONFIG["embed_batch_size"],
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype(np.float32)
print(f"Embeddings shape: {embeddings.shape}")
 
 
def flag_near_duplicates_greedy(records, embeddings, threshold, max_per_cluster=2):
    n = len(records)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
 
    k = min(20, n)
    scores, ids = index.search(embeddings, k)
 
    # Union-Find untuk cluster assignment
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(x, y):
        px, py = find(x), find(y)
        if px != py:
            parent[py] = px
 
    for i in range(n):
        for rank in range(1, k):
            j = int(ids[i][rank])
            sim = float(scores[i][rank])
            if sim < threshold:
                break
            union(i, j)
 
    # Kelompokkan per cluster
    clusters = defaultdict(list)
    for i in range(n):
        clusters[find(i)].append(i)
 
    # Per cluster: urutkan by track_a_score DESC, pertahankan max_per_cluster terbaik
    to_remove = set()
    for root, members in clusters.items():
        if len(members) <= max_per_cluster:
            continue
        members_sorted = sorted(
            members,
            key=lambda i: (records[i].get("track_a_score", 0.0), records[i].get("record_id", "")),
            reverse=True,
        )
        for i in members_sorted[max_per_cluster:]:
            to_remove.add(i)
 
    for i in to_remove:
        r = records[i]
        best_j = int(ids[i][1])
        r["flags"]["near_duplicate"] = True
        r["flags"]["duplicate_of"]   = records[best_j]["record_id"]
        r["flags"]["similarity"]      = round(float(scores[i][1]), 4)
 
    return records
 
random.seed(CONFIG["dedup_seed"])
raw_records = flag_near_duplicates_greedy(raw_records, embeddings, CONFIG["dedup_threshold"], CONFIG["dedup_max_per_cluster"])
n_dup = sum(1 for r in raw_records if r["flags"].get("near_duplicate"))
print(f"near_duplicate (greedy): {n_dup}")

modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Batches:   0%|          | 0/191 [00:00<?, ?it/s]

Embeddings shape: (24426, 768)
near_duplicate (greedy): 1104


### f. Soft score - track_a_score
#### high_similarity TIDAK dipakai untuk penalti (legitimate untuk factual_retrieval)

In [10]:
NUM_PATTERN = re.compile(r"\d+,\d+|\d+\.\d+(?!\d)|\b\d{2,}\b")
 
def compute_numeric_score(r):
    nums = NUM_PATTERN.findall(r.get("response",""))
    if not nums:
        return 1.0, []
    ctx = r.get("context","")
    unmatched = [n for n in nums if n not in ctx]
    score = 1.0 - (len(unmatched) / len(nums))
    return score, unmatched
 
def compute_leaked_phrases(r):
    text = f"{r.get('instruction','')} {r.get('response','')}".lower()
    return [p for p in LEAK_HIGH_CONFIDENCE if re.search(p, text, re.IGNORECASE)]
 
def compute_soft_scores(records):
    for r in records:
        num_score, unmatched = compute_numeric_score(r)
        leaked = compute_leaked_phrases(r)
        chunk_penalty = 0.2 if r["flags"].get("chunk_id_partial_invalid") else 0.0
        leak_penalty  = min(len(leaked) * 0.3, 1.0)
 
        r["numeric_score"]           = round(num_score, 4)
        r["flags"]["unmatched_numbers"] = unmatched
        if leaked:
            r["flags"]["leaked_phrase"] = leaked
        r["track_a_score"] = round(
            num_score * 0.5
            + (1.0 - leak_penalty) * 0.4
            + (1.0 - chunk_penalty) * 0.1,
            4,
        )
    return records
 
raw_records = compute_soft_scores(raw_records)
avg_score = sum(r["track_a_score"] for r in raw_records) / len(raw_records)
print(f"avg track_a_score: {avg_score:.4f}")

avg track_a_score: 0.9827


### g. PREVIEW per kategori flag

In [11]:
def preview_flagged(records, flag_name, reason_fn, n=CONFIG["preview_n"], seed=CONFIG["preview_seed"]):
    flagged = [r for r in records if r["flags"].get(flag_name)]
    print(f"\n{'='*60}\n{flag_name.upper()} — {len(flagged)} records ({len(flagged)/len(records)*100:.1f}%)\n{'='*60}")
    if not flagged:
        return
    sample = random.Random(seed).sample(flagged, min(n, len(flagged)))
    for r in sample:
        print(f"\nid={r['record_id']} | subtask={r.get('subtask_type')}")
        print(f"Q: {r.get('instruction','')[:200]}")
        print(f"A: {r.get('response','')[:200]}")
        print(f"Alasan: {reason_fn(r)}")
 
preview_flagged(raw_records, "schema_invalid",
                lambda r: r["flags"]["schema_invalid"])
preview_flagged(raw_records, "chunk_id_invalid",
                lambda r: f"≥50% chunk_id invalid: {r['flags']['missing_chunk_ids'][:3]}")
preview_flagged(raw_records, "chunk_id_partial_invalid",
                lambda r: f"sebagian invalid (soft penalti): {r['flags']['missing_chunk_ids'][:3]}")
preview_flagged(raw_records, "degenerate",
                lambda r: r["flags"]["degenerate"])
preview_flagged(raw_records, "table_ref_instruction",
                lambda r: r["flags"]["table_ref_instruction"])
preview_flagged(raw_records, "near_duplicate",
                lambda r: f"mirip dengan {r['flags']['duplicate_of']} (cos={r['flags']['similarity']:.3f})")
preview_flagged(raw_records, "leaked_phrase",
                lambda r: f"frasa bocor: {r['flags']['leaked_phrase']}")
preview_flagged(raw_records, "meta_ref_cleaned",
                lambda r: "meta-reference awalan sudah di-strip")


SCHEMA_INVALID — 0 records (0.0%)

CHUNK_ID_INVALID — 0 records (0.0%)

CHUNK_ID_PARTIAL_INVALID — 0 records (0.0%)

DEGENERATE — 20 records (0.1%)

id=9383213eac907d6d | subtask=factual_retrieval
Q: Berapa persentase IPM DI Yogyakarta?
A: IPM DI Yogyakarta sebesar 94,13.
Alasan: response kosong/terlalu pendek

id=cdaaa7a2fb332750 | subtask=factual_retrieval
Q: Berapa IPM Indonesia pada tahun 2022?
A: IPM 2022 tercatat sebesar 73,81.
Alasan: response kosong/terlalu pendek

id=a92344b41ebda334 | subtask=factual_retrieval
Q: Tabel nomor berapa yang menampilkan persentase penduduk berumur 0-6 tahun yang pernah/masih mengikuti pendidikan prasekolah di daerah perkotaan menurut provinsi dan jenis pendidikan prasekolah?
A: Tabel tersebut adalah Tabel 3.9.1.
Alasan: response kosong/terlalu pendek

id=6cd52cb893f648dc | subtask=source_navigation
Q: Di bagian mana saya bisa menemukan tabel yang membahas persentase penduduk perempuan berumur 7-23 tahun di daerah perdesaan menurut provinsi dan st

### h. DISTRIBUSI track_a_score per subtask

In [12]:
by_stratum = defaultdict(list)
for r in raw_records:
    by_stratum[r.get("subtask_type")].append(r["track_a_score"])
 
print(f"\n{'Subtask':<35}{'N':>7}{'Mean':>8}{'Min':>8}")
for sub, scores in sorted(by_stratum.items()):
    print(f"{sub:<35}{len(scores):>7}{np.mean(scores):>8.3f}{np.min(scores):>8.3f}")


Subtask                                  N    Mean     Min
causal_reasoning                      2822   0.984   0.380
comparative_analysis                  2828   0.977   0.500
definition_clarification              2883   0.992   0.380
document_overview                     1791   0.968   0.380
factual_retrieval                     3137   0.989   0.380
methodology_explanation               2775   0.984   0.500
multi_source_synthesis                2825   0.978   0.500
source_navigation                     2525   0.990   0.380
trend_interpretation                  2840   0.977   0.380


### i. FINALISASI — hard gate + export

In [13]:
HARD_GATE_KEYS = [
    "schema_invalid",
    "chunk_id_invalid",
    "degenerate",
    "table_ref_instruction",
    "near_duplicate",
]
 
filtered_records = [r for r in raw_records if not any(k in r["flags"] for k in HARD_GATE_KEYS)]
removed_records  = [r for r in raw_records if any(k in r["flags"] for k in HARD_GATE_KEYS)]
 
print(f"\nTotal raw       : {len(raw_records)}")
print(f"Lolos hard gate : {len(filtered_records)} ({len(filtered_records)/len(raw_records)*100:.1f}%)")
print(f"Gagal hard gate : {len(removed_records)} ({len(removed_records)/len(raw_records)*100:.1f}%)")
 
OUT = CONFIG["output_dir"]
FILTERED_PATH = f"{OUT}/filtered_records.jsonl"
REMOVED_PATH  = f"{OUT}/removed_records.jsonl"
REPORT_PATH   = f"{OUT}/qc_tahap1_report.json"
 
with open(FILTERED_PATH, "w", encoding="utf-8") as f:
    for r in filtered_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
 
with open(REMOVED_PATH, "w", encoding="utf-8") as f:
    for r in removed_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
 
removal_reasons = defaultdict(int)
for r in removed_records:
    for k in HARD_GATE_KEYS:
        if k in r["flags"]:
            removal_reasons[k] += 1
 
report = {
    "timestamp":    datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "total_raw":    len(raw_records),
    "total_filtered": len(filtered_records),
    "total_removed":  len(removed_records),
    "n_meta_ref_cleaned": n_cleaned,
    "removal_reasons": dict(removal_reasons),
    "track_a_score_by_subtask": {
        s: {"n": len(v), "mean": float(np.mean(v)), "min": float(np.min(v))}
        for s, v in by_stratum.items()
    },
    "config": {k: v for k, v in CONFIG.items() if k != "skip_section_sources"},
}
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
 
print(f"\nRemoval reasons:")
for k, v in removal_reasons.items():
    print(f"  {k:<25}: {v}")
print(f"\nOutput:")
print(f"  {FILTERED_PATH}")
print(f"  {REMOVED_PATH}")
print(f"  {REPORT_PATH}")


Total raw       : 24426
Lolos hard gate : 23001 (94.2%)
Gagal hard gate : 1425 (5.8%)

Removal reasons:
  near_duplicate           : 1104
  table_ref_instruction    : 501
  degenerate               : 20

Output:
  output_05b/filtered_records.jsonl
  output_05b/removed_records.jsonl
  output_05b/qc_tahap1_report.json
